# Homework #5: Model Deployment

**Instructions:** Using the F1 dataset, build a predictive model and log it in MLflow and write the ML model predictions into a database.

1. [`20 pts`] Create two (2) new tables in your own database where you'll store the predictions from each model
2. [`30 pts`] Build two (2) predictive models using MLflow, logging hyperparameters, the model itself, four metrics, and two artifacts
3. [`30 pts`] For each model, store its predictions in the corresponding table in your own database
4. [`20 pts`] Push your code to GitHub upon completion

## Step 1 — Imports

In [0]:
# COMMAND ----------
%pip install typing_extensions==4.12.2 mlflow --upgrade

In [0]:
%pip install mlflow scikit-learn matplotlib pandas numpy typing_extensions>=4.10.0

In [0]:
import sys
if 'typing_extensions' in sys.modules:
    del sys.modules['typing_extensions']

import mlflow
import mlflow.sklearn
import tempfile
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
    median_absolute_error
)

print("All imports successful.")
print(f"MLflow version: {mlflow.__version__}")

## Step 2 — Load & Prepare the F1 Data

In [0]:
DATA_DIR = "/Volumes/gr5069/raw/f1_data/"

results    = spark.read.csv(DATA_DIR + "results.csv",    header=True, inferSchema=True, nullValue="\\N").toPandas()
races      = spark.read.csv(DATA_DIR + "races.csv",      header=True, inferSchema=True, nullValue="\\N").toPandas()
qualifying = spark.read.csv(DATA_DIR + "qualifying.csv", header=True, inferSchema=True, nullValue="\\N").toPandas()

print("Shapes:")
for name, df in [("results", results), ("races", races), ("qualifying", qualifying)]:
    print(f"  {name:15s}: {df.shape}")

In [0]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

df_pd = results.copy()
df_pd = df_pd.replace("\\N", np.nan)

selected_cols = [
    "grid",
    "laps",
    "fastestLap",
    "rank",
    "fastestLapSpeed",
    "points",
    "positionOrder"
]

df_model = df_pd[selected_cols].copy()

for col in df_model.columns:
    df_model[col] = pd.to_numeric(df_model[col], errors="coerce")

df_model = df_model.dropna()

# keep raceId + driverId for storing predictions later
df_meta = df_pd[["raceId", "driverId"] + selected_cols].copy()
for col in selected_cols:
    df_meta[col] = pd.to_numeric(df_meta[col], errors="coerce")
df_meta = df_meta.dropna()

X = df_model.drop("positionOrder", axis=1)
y = df_model["positionOrder"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_meta_full = df_meta.drop("positionOrder", axis=1)
y_meta_full = df_meta["positionOrder"]
X_meta_train, X_meta_test, _, _ = train_test_split(
    X_meta_full, y_meta_full, test_size=0.2, random_state=42
)

print("Dataset shape:", df_model.shape)
print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")
display(df_model.head())

## Step 3 — Set MLflow Experiment

In [0]:
import mlflow

mlflow.set_experiment("/Shared/F1-MLflow-Experiment")

## Step 4 — Training Function

Logs: hyperparameters, the model, **4 metrics** (MSE, MAE, R², Median AE), **2 artifacts** (feature importance CSV + residual plot).

In [0]:
import mlflow
import mlflow.sklearn
import tempfile
import matplotlib.pyplot as plt

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, median_absolute_error

def run_experiment(model, params, run_name):

    with mlflow.start_run(run_name=run_name) as run:

        # train model
        model.set_params(**params)
        model.fit(X_train, y_train)
        preds = model.predict(X_test)

        # log params
        for k, v in params.items():
            mlflow.log_param(k, v)

        # 4 metrics
        mse    = mean_squared_error(y_test, preds)
        mae    = mean_absolute_error(y_test, preds)
        r2     = r2_score(y_test, preds)
        med_ae = median_absolute_error(y_test, preds)

        mlflow.log_metric("mse",            mse)
        mlflow.log_metric("mae",            mae)
        mlflow.log_metric("r2",             r2)
        mlflow.log_metric("median_abs_err", med_ae)

        # log model
        mlflow.sklearn.log_model(model, f"{run_name}_model")

        # artifact 1: feature importance csv
        importance_df = pd.DataFrame({
            "feature":    X_train.columns,
            "importance": model.feature_importances_
        }).sort_values("importance", ascending=False)

        tmp_csv = tempfile.NamedTemporaryFile(delete=False, suffix=".csv")
        importance_df.to_csv(tmp_csv.name, index=False)
        mlflow.log_artifact(tmp_csv.name, artifact_path="artifacts")

        # artifact 2: residual plot
        residuals = y_test - preds

        plt.figure(figsize=(8, 5))
        plt.scatter(preds, residuals, alpha=0.4)
        plt.axhline(y=0, linestyle="--", color="red")
        plt.xlabel("Predicted Position Order")
        plt.ylabel("Residuals")
        plt.title(f"{run_name} — Residual Plot")

        tmp_png = tempfile.NamedTemporaryFile(delete=False, suffix=".png")
        plt.savefig(tmp_png.name, bbox_inches="tight")
        mlflow.log_artifact(tmp_png.name, artifact_path="artifacts")
        plt.close()

        print(f"{run_name} | mse={mse:.4f} | mae={mae:.4f} | r2={r2:.4f} | med_ae={med_ae:.4f}")

        return preds, run.info.run_id

## Step 5 — Run Both Models

In [0]:
from sklearn.ensemble import RandomForestRegressor

rf_params = {
    "n_estimators":      200,
    "max_depth":         10,
    "min_samples_split": 5,
    "random_state":      42
}

preds_rf, run_id_rf = run_experiment(
    RandomForestRegressor(), rf_params, run_name="RandomForest"
)
print(f"RF Run ID: {run_id_rf}")

In [0]:
from sklearn.ensemble import GradientBoostingRegressor

gb_params = {
    "n_estimators":  200,
    "learning_rate": 0.05,
    "max_depth":     5,
    "subsample":     0.8,
    "random_state":  42
}

preds_gb, run_id_gb = run_experiment(
    GradientBoostingRegressor(), gb_params, run_name="GradientBoosting"
)
print(f"GB Run ID: {run_id_gb}")

## Step 6 — [20 pts] Create Database Tables

Using Databricks `spark.sql()` — no `sqlalchemy` needed.

In [0]:
YOUR_SCHEMA = "default"

TABLE_RF = f"workspace.{YOUR_SCHEMA}.f1_predictions_random_forest"
TABLE_GB = f"workspace.{YOUR_SCHEMA}.f1_predictions_gradient_boosting"

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {TABLE_RF} (
        race_id       INT,
        driver_id     INT,
        predicted_pos DOUBLE,
        actual_pos    DOUBLE
    )
""")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {TABLE_GB} (
        race_id       INT,
        driver_id     INT,
        predicted_pos DOUBLE,
        actual_pos    DOUBLE
    )
""")

print(f"Created: {TABLE_RF}")
print(f"Created: {TABLE_GB}")

## Step 7 — [30 pts] Write Predictions to Database

In [0]:
# Build prediction DataFrames
test_meta = X_meta_test[["raceId", "driverId"]].copy().reset_index(drop=True)

def build_pred_df(preds, test_meta, y_test):
    out = test_meta.copy()
    out["predicted_pos"] = preds
    out["actual_pos"]    = y_test.values
    out.columns         = ["race_id", "driver_id", "predicted_pos", "actual_pos"]
    return out

pred_rf_df = build_pred_df(preds_rf, test_meta, y_test)
pred_gb_df = build_pred_df(preds_gb, test_meta, y_test)

pred_rf_df.head(3)

%md
## Step 8 — [30 pts] Store Predictions to Database

In [0]:
# Write to Delta tables via Spark
spark.createDataFrame(pred_rf_df).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(TABLE_RF)
spark.createDataFrame(pred_gb_df).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(TABLE_GB)

print(f"Written {len(pred_rf_df)} rows to {TABLE_RF}")
print(f"Written {len(pred_gb_df)} rows to {TABLE_GB}")

In [0]:
# Verify
spark.sql(f"SELECT COUNT(*) AS n FROM {TABLE_RF}").show()
spark.sql(f"SELECT COUNT(*) AS n FROM {TABLE_GB}").show()
spark.sql(f"SELECT * FROM {TABLE_RF} LIMIT 5").show()